In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# parse payload
COPY parse_payload.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow
fsspec
s3fs

pandas
tqdm
xmltodict
boto3

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from parse_payload import PayloadToDataFrame
import time
from tqdm import tqdm
import boto3

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

# get the idx
int_idx_array = int(os.environ['AWS_BATCH_JOB_ARRAY_INDEX'])
print(f'Index: {int_idx_array}')

# constants
str_project = '20241112-simple-model-test'
str_task = '02_parse_payloads'
str_datecol = 'REQUEST_DATETIME'

# get the list of filenames
print('Getting list of filenames...')
str_filename = 'df_list_str_filename_2.csv'
str_local_path = f'./{str_filename}'
download_from_s3(
    str_local_path=str_local_path,
    str_bucket_path=f'filenames_for_parsing/{str_filename}',
    str_project=str_project,
)
df = pd.read_csv(str_local_path)
list_str_filename = list(df['str_filename'])

# get the filename to parse
str_filename = list_str_filename[int_idx_array]
print(f'Parsing File: {str_filename}')
                
# read file
print('Imporing file...')
str_uri = f's3://20241022-parse-snowflake-payloads/01_pull_payloads/{str_filename}'
df = pd.read_parquet(str_uri)

# init
cls_parse_payload = PayloadToDataFrame()

# iterate through rows and extract raw data
list_df_tmp = []
for a, str_request in enumerate(tqdm(df['REQUEST_JSON'])):
    # replace NaN
    time_start = time.perf_counter()
    
    # get the model name
    str_model_name = df['RESPONSE_MODEL_NAME'].iloc[a]
    
    # get data
    df_tmp = cls_parse_payload.get_data(str_request=str_request)

    # make copy
    df_tmp = df_tmp.copy()

    # assign
    df_tmp['FILE_NAME'] = df['FILE_NAME'].iloc[a]
    df_tmp['ACCOUNTID'] = df['ACCOUNTID'].iloc[a]
    df_tmp['REQUEST_DATETIME'] = df['REQUEST_DATETIME'].iloc[a]
    df_tmp['REQUEST_ID'] = df['REQUEST_ID'].iloc[a]
    df_tmp['RESPONSE_MODEL_NAME'] = df['RESPONSE_MODEL_NAME'].iloc[a]
    
    # get number of rows
    int_nrows = df_tmp.shape[0]
    
    # logic
    if int_nrows == 1:
        list_bitdebtor = [1]
    else:
        list_bitdebtor = [1, 0]
        
    # assign
    df_tmp['BITDEBTOR'] = list_bitdebtor
    
    # reorder
    list_cols_id = [
        'FILE_NAME',
        'ACCOUNTID',
        'REQUEST_DATETIME',
        'REQUEST_ID',
        'RESPONSE_MODEL_NAME',
        'BITDEBTOR',
    ]
    list_cols = [col for col in df_tmp.columns if col not in list_cols_id]
    list_cols = list_cols_id + list_cols
    df_tmp = df_tmp[list_cols].copy()

    # end time
    time_end = time.perf_counter()
    # sec
    flt_sec = time_end - time_start

    # assign
    df_tmp['flt_sec'] = flt_sec
    
    # append
    list_df_tmp.append(df_tmp)

# make df
df = pd.concat(list_df_tmp)

# convert date to datetime
df[str_datecol] = pd.to_datetime(df[str_datecol])

# identify the non-numeric columns
list_cols = []
for col in df.columns:
    str_dtype = df[col].dtype
    if str_dtype not in ['int64','float64']:
        list_cols.append(col)
    else:
        pass
int_n_cols = len(list_cols)
print(f'There are {int_n_cols} possible non-numeric columns')

# remove dates
list_cols = [col for col in list_cols if col != str_datecol]

# convert non-numeric to string
for col in tqdm(list_cols):
    try:
        # convert to string
        df[col] = df[col].astype(str)
    except:
        pass

# save
str_filename_date = str_filename.split('_')[2]
str_filename = f'df_parsed_{str_filename_date}'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=simple-model-test-parse-snowflake

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 403B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.9
#2 DONE 0.3s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/7] FROM docker.io/library/python:3.9@sha256:332741499f49a3f3e7749dad70e6ecf1129f00a269fdd6111da2ed2693fbe50e
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 13.29kB done
#5 DONE 0.0s

#6 [4/7] COPY requirements.txt .
#6 CACHED

#7 [3/7] RUN pip install --upgrade pip
#7 CACHED

#8 [2/7] RUN apt-get update
#8 CACHED

#9 [5/7] RUN pip install -r requirements.txt
#9 CACHED

#10 [6/7] COPY script.py .
#10 DONE 0.0s

#11 [7/7] COPY parse_payload.py .
#11 DONE 0.0s

#12 exporting to image
#12 exporting layers 0.0s done
#12 writing image sha256:53a7ee473318105dd6f1073bb5e315a5ab2df233bc2c2e5189b9106c7b006ed3 done
#12 naming to docker.io/library/simple-mo

Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'simple-model-test-parse-snowflake' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/simple-model-test-parse-snowflake]
b37c391e7297: Preparing
1dc39095c546: Preparing
a4e0c3178a26: Preparing
e04c134fd65c: Preparing
d152368bac3d: Preparing
1a8bcd41b0c3: Preparing
6f48ad948751: Preparing
1354cd991b1c: Preparing
68fff311c877: Preparing
96d99c63b722: Preparing
00547dd240c4: Preparing
b6ca42156b9f: Preparing
24b5ce0f1e07: Preparing
1a8bcd41b0c3: Waiting
6f48ad948751: Waiting
1354cd991b1c: Waiting
68fff311c877: Waiting
b6ca42156b9f: Waiting
96d99c63b722: Waiting
24b5ce0f1e07: Waiting
00547dd240c4: Waiting
b37c391e7297: Layer already exists
d152368bac3d: Layer already exists
e04c134fd65c: Layer already exists
a4e0c3178a26: Layer already exists
1a8bcd41b0c3: Layer already exists
6f48ad948751: Layer already exists
68fff311c877: Layer already exists
1354cd991b1c: Layer already exists
96d99c63b722: Layer already exists
00547dd240c4: Layer already exists
24b5ce0f1e07: Layer already exists
b6ca42156b9f: La

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass